# MovieLens データセットを用いたハイブリッドレコメンデーション

__注:__ この __als_bqml_hybrid.ipynb__ ノートブックに進む前に、関連する __als_bqml.ipynb__ ノートブックを完了することをお勧めします。ただし、データセットとトレーニング済みモデルを引き継ぐオプションがあるため、このラボでは必須ではありません。すでに MovieLens データセットとトレーニング済みモデルをお持ちの場合は、「データセットとトレーニング済みモデルのインポート」セクションをスキップできます。

## 学習目標
1. BigQuery 行列分解（Matrix Factorization）モデルからユーザー因子とプロダクト因子を抽出する方法を理解する
2. BigQuery ハイブリッドレコメンデーションモデルの入力をフォーマットする方法を理解する

In [ ]:
PROJECT = !(gcloud config get-value core/project)
PROJECT = PROJECT[0]

%env PROJECT=$PROJECT

%load_ext google.cloud.bigquery

## データセットとトレーニング済みモデルのインポート
前のノートブックでは、2,000 万件の映画レコメンデーションをインポートし、BigQuery ML で ALS モデルをトレーニングしました。

同じテーブルを使用しますが、新しい環境の場合は、以下のコマンドを実行してクリーンなデータをコピーしてください。

まず BigQuery データセットを作成し、データをコピーします

In [ ]:
!bq mk movielens

In [ ]:
%%bash
rm -r bqml_data
mkdir bqml_data
cd bqml_data
curl -O 'http://files.grouplens.org/datasets/movielens/ml-20m.zip'
unzip ml-20m.zip
yes | bq rm -r $PROJECT:movielens
bq --location=US mk --dataset \
    --description 'Movie Recommendations' \
    $PROJECT:movielens
bq --location=US load --source_format=CSV \
    --autodetect movielens.ratings ml-20m/ratings.csv
bq --location=US load  --source_format=CSV \
    --autodetect movielens.movies_raw ml-20m/movies.csv

そしてクリーンアップされた `movielens.movies` テーブルを作成します。

In [ ]:
%%bigquery --project $PROJECT
CREATE OR REPLACE TABLE movielens.movies AS
    SELECT * REPLACE(SPLIT(genres, "|") AS genres)
    FROM movielens.movies_raw

次に、トレーニング済みのレコメンデーションモデルをコピーします。プロジェクトが EU にある場合は、以下のロケーションを US から EU に変更する必要がある点に注意してください。執筆時点では、`bq cp` を使用してリージョンをまたいでモデルをコピーすることはできません。

In [ ]:
%%bash
bq --location=US cp \
cloud-training-demos:movielens.recommender \
movielens.recommender

次に、映画レコメンデーションの予測を呼び出して、モデルがまだ動作することを確認します:

In [ ]:
%%bigquery --project $PROJECT
SELECT * FROM
ML.PREDICT(MODEL `movielens.recommender`, (
  SELECT 
    movieId, title, 903 AS userId
  FROM movielens.movies, UNNEST(genres) g
  WHERE g = 'Comedy'
))
ORDER BY predicted_rating DESC
LIMIT 5

### ユーザー情報と映画情報の組み込み
行列分解アプローチでは、評価行列から利用可能な情報以外にユーザーや映画に関する情報を一切使用しません。しかし、多くの場合、ユーザー情報（居住都市、年収、年間支出額など）があり、カタログ内の製品に関する詳細情報もほぼ常に存在します。レコメンデーションモデルにこの情報をどのように組み込むべきでしょうか？

その答えは、行列分解アプローチから得られるユーザー因子とプロダクト因子が、評価行列から得られるユーザーや製品に関する情報の簡潔な表現になっていることを認識することにあります。この情報を利用可能な他の情報と連結し、評価を予測する回帰モデルをトレーニングできます。
### ユーザー因子とプロダクト因子の取得
ML.WEIGHTS からユーザー因子またはプロダクト因子を取得できます。たとえば、movieId=96481 のプロダクト因子と userId=54192 のユーザー因子を取得するには、次のように実行します:

In [ ]:
%%bigquery --project $PROJECT
SELECT 
    processed_input,
    feature,
    TO_JSON_STRING(factor_weights) AS factor_weights,
    intercept
FROM ML.WEIGHTS(MODEL `movielens.recommender`)
WHERE
    (processed_input = 'movieId' AND feature = '96481')
    OR (processed_input = 'userId' AND feature = '54192')

これらの重みを掛け合わせて切片を加えることで、行列分解アプローチにおいてこの movieId と userId の組み合わせの予測評価が得られます。

これらの重みは、映画とユーザーの行動の低次元表現としても機能します。ユーザー因子、プロダクト因子、およびユーザーや製品について把握しているその他の情報が与えられた場合に、評価を予測する回帰モデルを作成できます。
### 入力特徴量の作成
MovieLens データセットにはユーザー情報がなく、映画自体に関する情報もほとんどありません。そこで、概念を説明するために、ユーザーに関する合成情報を作成してみましょう:

In [ ]:
%%bigquery --project $PROJECT
CREATE OR REPLACE TABLE movielens.users AS
SELECT
    userId,
    RAND() * COUNT(rating) AS loyalty,
    CONCAT(SUBSTR(CAST(userId AS STRING), 0, 2)) AS postcode
FROM
  movielens.ratings
GROUP BY userId

ユーザーに関する入力特徴量は、ユーザーテーブルと ML 重みを結合し、すべてのユーザー情報と重み配列からのユーザー因子を選択することによって取得できます。

In [ ]:
%%bigquery --project $PROJECT
WITH userFeatures AS (
  SELECT 
     u.*,
     (SELECT ARRAY_AGG(weight) FROM UNNEST(factor_weights)) AS user_factors
  FROM movielens.users u
  JOIN ML.WEIGHTS(MODEL movielens.recommender) w
  ON processed_input = 'userId' AND feature = CAST(u.userId AS STRING)
)

SELECT * FROM userFeatures
LIMIT 5

同様に、映画データのプロダクト特徴量を取得できますが、映画には複数のジャンルが含まれる可能性があるため、ジャンルの処理方法を決定する必要があります。ジャンルごとに別々のトレーニング行を作成することにした場合、プロダクト特徴量を次のように構築できます。

In [ ]:
%%bigquery --project $PROJECT
WITH productFeatures AS (
  SELECT 
      p.* EXCEPT(genres),
      g, (SELECT ARRAY_AGG(weight) FROM UNNEST(factor_weights))
            AS product_factors
  FROM movielens.movies p, UNNEST(genres) g
  JOIN ML.WEIGHTS(MODEL movielens.recommender) w
  ON processed_input = 'movieId' AND feature = CAST(p.movieId AS STRING)
)

SELECT * FROM productFeatures
LIMIT 5

これら 2 つの WITH 句を組み合わせ、movieId と userId の組み合わせに対応する評価（評価テーブルに存在する場合）を取り込むことで、トレーニングデータセットを作成できます。

**TODO 1**: 上記の 2 つのクエリを組み合わせて、各評価のユーザー因子とプロダクト因子を取得します。

In [ ]:
%%bigquery --project $PROJECT
CREATE OR REPLACE TABLE movielens.hybrid_dataset AS

    WITH userFeatures AS (
      # TODO: ここにユーザー特徴量クエリを記述します
    ),

    productFeatures AS (
      # TODO: ここにプロダクト特徴量クエリを記述します
    )

    SELECT
        p.* EXCEPT(movieId),
        u.* EXCEPT(userId),
        rating 
    FROM productFeatures p, userFeatures u
    JOIN movielens.ratings r
    ON r.movieId = p.movieId AND r.userId = u.userId

このテーブルの行の 1 つは次のようになります:

In [ ]:
%%bigquery --project $PROJECT
SELECT *
FROM movielens.hybrid_dataset
LIMIT 1

要約すると、映画に関するいくつかの属性、映画に対応するプロダクト因子配列、ユーザーに関するいくつかの属性、およびユーザーに対応するユーザー因子配列が得られます。これらが行列分解モデルを基にしてユーザーや映画に関するメタデータを追加した「ハイブリッド」レコメンデーションモデルへの入力を形成します。
### ハイブリッドレコメンデーションモデルのトレーニング
執筆時点では、BigQuery ML は回帰モデルへの入力として配列を処理できません。そのため、配列を配列要素がフィールドとなる struct に変換する関数を定義しましょう:

In [ ]:
%%bigquery --project $PROJECT
CREATE OR REPLACE FUNCTION movielens.arr_to_input_16_users(u ARRAY<FLOAT64>)
RETURNS 
    STRUCT<
        u1 FLOAT64,
        u2 FLOAT64,
        u3 FLOAT64,
        u4 FLOAT64,
        u5 FLOAT64,
        u6 FLOAT64,
        u7 FLOAT64,
        u8 FLOAT64,
        u9 FLOAT64,
        u10 FLOAT64,
        u11 FLOAT64,
        u12 FLOAT64,
        u13 FLOAT64,
        u14 FLOAT64,
        u15 FLOAT64,
        u16 FLOAT64
    > AS (STRUCT(
        u[OFFSET(0)],
        u[OFFSET(1)],
        u[OFFSET(2)],
        u[OFFSET(3)],
        u[OFFSET(4)],
        u[OFFSET(5)],
        u[OFFSET(6)],
        u[OFFSET(7)],
        u[OFFSET(8)],
        u[OFFSET(9)],
        u[OFFSET(10)],
        u[OFFSET(11)],
        u[OFFSET(12)],
        u[OFFSET(13)],
        u[OFFSET(14)],
        u[OFFSET(15)]
));

これにより以下が得られます:

In [ ]:
%%bigquery --project $PROJECT
SELECT movielens.arr_to_input_16_users(u).*
FROM (SELECT
    [0., 1., 2., 3., 4., 5., 6., 7., 8., 9., 10., 11., 12., 13., 14., 15.] AS u)

プロダクト因子の配列を名前付き列に変換するために、`movielens.arr_to_input_16_products` という同様の関数を作成できます。

**TODO 2**: サイズ 16 のプロダクト因子配列から名前付き列を返す関数を作成します。

In [ ]:
%%bigquery --project $PROJECT
CREATE OR REPLACE FUNCTION movielens.arr_to_input_16_products(p ARRAY<FLOAT64>)
RETURNS 
    STRUCT<
        p1 FLOAT64,
        p2 FLOAT64,
        p3 FLOAT64,
        p4 FLOAT64,
        p5 FLOAT64,
        p6 FLOAT64,
        p7 FLOAT64,
        p8 FLOAT64,
        p9 FLOAT64,
        p10 FLOAT64,
        p11 FLOAT64,
        p12 FLOAT64,
        # TODO: この struct の構築を完成させてください
    > AS (STRUCT(
        p[OFFSET(0)],
        p[OFFSET(1)],
        p[OFFSET(2)],
        p[OFFSET(3)],
        p[OFFSET(4)],
        p[OFFSET(5)],
        p[OFFSET(6)],
        p[OFFSET(7)],
        p[OFFSET(8)],
        p[OFFSET(9)],
        p[OFFSET(10)],
        p[OFFSET(11)],
        p[OFFSET(12)],
        # TODO: この struct の構築を完成させてください
));

次に、ユーザーや製品に関するメタデータを行列分解アプローチから得られたユーザー因子およびプロダクト因子と結び付け、評価を予測する回帰モデルを作成できます:

In [ ]:
%%bigquery --project $PROJECT
CREATE OR REPLACE MODEL movielens.recommender_hybrid 
OPTIONS(model_type='linear_reg', input_label_cols=['rating'])
AS

SELECT
  * EXCEPT(user_factors, product_factors),
    movielens.arr_to_input_16_users(user_factors).*,
    movielens.arr_to_input_16_products(product_factors).*
FROM
  movielens.hybrid_dataset

トレーニングデータセットの作成に使用したユーザー情報は架空のものであるため（loyalty 列の作成における RAND() に注意）、このモデルの評価指標を確認する意味はありません — この演習はどのように行えるかを実演するために実施しました。もちろん、より高度なモデルが必要な場合は、dnn_regressor モデルをトレーニングしてハイパーパラメータを最適化することもできます。しかし、そこまで行うのであれば、[AutoML Tables](https://cloud.google.com/automl-tables) の使用を検討する方が良いかもしれません。

Copyright 2022 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.